Imports needed to load the model and generate text.

In [2]:
import random, pickle

import numpy as np

import tensorflow as tf
from tensorflow.keras.models import load_model

Load the trained model, vocabulary, and tokenizer.

In [4]:
from nltk.tokenize import RegexpTokenizer

model = load_model('../models/keras_next_word_model.keras')

with open('../data/processed/unique_tokens.p', 'rb') as f:
    unique_tokens = pickle.load(f)

with open('../data/processed/unique_token_idx.p', 'rb') as f:
    unique_token_idx = pickle.load(f)

tokenizer = RegexpTokenizer(r'\w+')

Predict the most likely next words for a short phrase.

In [3]:
def predict_next_word(input_text, n_best=1):
    words = input_text.lower().split()[-10:]

    token_ids = [unique_token_idx[word] for word in words]
    token_ids = [0] * (10 - len(token_ids)) + token_ids

    X = np.array([token_ids])

    predictions = model.predict(X, verbose=0)[0]

    best_indices = np.argsort(predictions)[-n_best:][::-1]

    return best_indices

Show the top five predictions after "I will".

In [4]:
possible = predict_next_word("I will", 5)

for idx in possible:
    print(unique_tokens[idx])

be
have
help
only
take


Repeat prediction to build a longer text continuation.

In [5]:
def generate_text(input_text, n_words, creativity=3):
    word_sequence = input_text.split()
    current = 0
    for _ in range(n_words):
        sub_sequence = " ".join(tokenizer.tokenize(" ".join(word_sequence).lower())[current:current+n_words])
        try:
            choice = unique_tokens[random.choice(predict_next_word(sub_sequence, creativity))]
        except:
            choice = random.choice(unique_tokens)
        word_sequence.append(choice)
        current += 1
    return " ".join(word_sequence)

Generate fifteen words starting from "I will".

In [6]:
generate_text("I will", 15, 2)

'I will be a military text q positions on wednesday january claimed are reportedly killed on facebook'